In [ ]:
# 1. Import des bibliothèques et configuration initiale
import json
from datetime import date, timedelta
from pathlib import Path

class FakeRequest:
    def __init__(self, method='GET', post=None, get=None):
        self.method = method
        self.POST = post or {}
        self.GET = get or {}
        self.session = {}

class FakeFace:
    def __init__(self, pk, label='A', support=None):
        self.pk = pk
        self.label = label
        self.support = support
        self.id = pk

class FakeSupport:
    def __init__(self, pk, code='SUP-001', ville='Ouagadougou', categorie='Standard', format_code='4x3', nom='Panneau', quartier='Centre'):
        self.pk = pk
        self.code = code
        self.ville = ville
        self.categorie = categorie
        self.format = format_code
        self.nom = nom
        self.quartier = quartier
        self.support_id = pk

context = {
    'staff_mode': False,
    'shared_edit_mode': False,
    'editing_request': False,
    'allowed_categories': ['Standard'],
    'initial_selected_faces': [
        {'id': '101', 'code': 'SUP-001', 'label': 'Face A', 'quartier': 'Centre', 'support_id': '1'}
    ],
    'dates': {'date_debut': '2026-01-10', 'date_fin': '2026-01-15'},
}
print('Contexte initialisé pour la page Étape 1.')

"# 2. Chargement des filtres et des supports disponibles
villes = ['Ouagadougou', 'Bobo-Dioulasso', 'Koudougou']
categories = ['Standard', 'Premium', 'Écran']
formats = ['4x3', '6x4', '8x5']

supports = [
    FakeSupport(1, 'SUP-001', 'Ouagadougou', 'Standard', '4x3', 'Panneau Centre', 'Centre'),
    FakeSupport(2, 'SUP-002', 'Ouagadougou', 'Premium', '6x4', 'Panneau Grand', 'Kadiogo'),
    FakeSupport(3, 'SUP-003', 'Bobo-Dioulasso', 'Standard', '4x3', 'Panneau Lafi', 'Centre'),
]


def filter_supports(supports_list, ville=None, categorie=None, format_code=None):
    result = supports_list
    if ville:
        result = [s for s in result if s.ville == ville]
    if categorie:
        result = [s for s in result if s.categorie == categorie]
    if format_code:
        result = [s for s in result if s.format == format_code]
    return result

print('Supports filtrés:', filter_supports(supports, ville='Ouagadougou', categorie='Standard'))

"# 3. Validation des dates et calcul de la durée

def diff_days_inclusive(start, end):
    start_date = date.fromisoformat(start)
    end_date = date.fromisoformat(end)
    return (end_date - start_date).days + 1


def validate_dates(date_debut, date_fin):
    if not date_debut or not date_fin:
        return False, 'Dates manquantes'
    if date_fin <= date_debut:
        return False, 'La date de fin doit être après la date de début'
    if date_debut < date.today():
        return False, 'La date de début ne peut pas être dans le passé'
    return True, diff_days_inclusive(date_debut, date_fin)

print(validate_dates('2026-01-10', '2026-01-15'))

"# 4. Gestion du mode client / staff et du formulaire

def build_form_data(mode='public', type_client='nouveau'):
    payload = {
        'faces_selectionnees': json.dumps(['101']),
        'supports_selectionnees': '[]',
        'date_debut': '2026-01-10',
        'date_fin': '2026-01-15',
        'type_client': type_client,
    }
    if mode == 'staff':
        payload.update({
            'nom_contact': 'Awa Contact',
            'societe': 'Société Demo',
            'email': 'awa@example.com',
            'telephone': '70000000',
            'accepte_contact': 'on',
        })
    return payload

print(build_form_data(mode='staff', type_client='existant'))

"# 5. Construction de la vue liste et des cartes

def build_list_group(supports_list):
    by_ville = {}
    for support in supports_list:
        by_ville.setdefault(support.ville, []).append(support)
    return by_ville

print(build_list_group(supports))

"# 6. Sélection des faces et panier
panier = {}

def add_face(face_id, code):
    panier[face_id] = {'code': code}
    return panier


def remove_face(face_id):
    panier.pop(face_id, None)
    return panier

add_face('101', 'SUP-001')
print('Panier après ajout:', panier)
remove_face('101')
print('Panier après retrait:', panier)

"# 7. Contrôle des disponibilités et états par face

def face_status(date_debut, date_fin, face_is_free=True):
    if not face_is_free:
        return 'occupe'
    return 'libre'

print(face_status('2026-01-10', '2026-01-15', True))
print(face_status('2026-01-10', '2026-01-15', False))

"# 8. Tarification et estimation

def estimate_total(tarifs, count_faces, nb_days):
    months = max(1, (nb_days + 29) // 30)
    total = 0
    for code, price in tarifs.items():
        total += float(price) * count_faces * months
    return total

print(estimate_total({'4x3': 12000}, 2, 45))

"# 9. Soumission du formulaire étape 1

def prepare_submission(selected_faces, supports, date_debut, date_fin):
    payload = {
        'faces_selectionnees': json.dumps(selected_faces),
        'supports_selectionnees': json.dumps(supports),
        'date_debut': date_debut,
        'date_fin': date_fin,
    }
    return payload

print(prepare_submission(['101', '102'], [], '2026-01-10', '2026-01-15'))


# Notebook – Logique de la page de réservation Étape 1

Plan de travail: filtres, dates, sélection d'emplacements, validation et préparation du payload.
